# Makemore Part 5: WaveNet-Style Hierarchical Character Models

In this notebook we take the multilayer perceptron from earlier and reshape it into a **hierarchical, WaveNet-style** character-level language model. Instead of flattening the whole context window into one giant vector, we process it in stages: first pairs of characters, then pairs of pairs, and so on. This is the same idea that powers modern autoregressive models.

## Learning objectives
- Understand why a flat MLP wastes representational power for long contexts.
- Build a hierarchical model by stacking `FlattenConsecutive`, `Linear`, `BatchNorm1d`, and `Tanh` layers.
- See how modern layers such as Kaiming initialization and BatchNorm make deep networks trainable.
- Train the model with cross-entropy and evaluate it on train/validation/test splits.
- Sample new names from the trained model.

## Why this matters
Language has structure at many scales: phonemes form syllables, syllables form morphemes, morphemes form words. A flat model sees the whole input at once and must learn all of that structure in one weight matrix. A hierarchical model learns short-range patterns first and composes them into longer-range patterns, exactly the inductive bias we want for sequences.

## Prerequisites
- [Makemore Part 1: Bigrams](../notebooks/03_makemore_bigram.ipynb) — character-level modeling and the bigram approach.
- [Makemore Part 3: MLP](../notebooks/05_makemore_mlp.ipynb) — embeddings, hidden layers, and cross-entropy loss.
- Basic PyTorch: tensors, `torch.nn.functional.cross_entropy`, and automatic differentiation.
- The `nnzero` package installed in this repo (it contains the same tiny layers we built by hand earlier).

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Tiny layers and helpers from the course repo.
# These mirror the from-scratch implementations in earlier lectures, packaged
# so we can focus on the architecture in this notebook.
from nnzero import Linear, BatchNorm1d, Tanh, Embedding, FlattenConsecutive, Sequential
from nnzero.utils import build_vocab, build_dataset, split_dataset, load_names, set_seed

%matplotlib inline

## Quick-run mode for CI

When the environment variable `NNZERO_QUICK_RUN=1` is set, training loops run for only a small number of steps so that notebooks can be validated quickly. Students should leave this unset to train the full models.

In [ ]:
import os
QUICK_RUN = os.environ.get("NNZERO_QUICK_RUN", "0") == "1"
if QUICK_RUN:
    print("Quick-run mode enabled: training loops will use far fewer steps.")

## Load the dataset

We use the same list of ~32k names as in the earlier makemore lectures. The helper `load_names` reads from `data/names.txt` relative to the repo root.

In [ ]:
# Load all names from the repo's data directory
words = load_names('data/names.txt')
print(f"Total names: {len(words)}")
print(f"Longest name: {max(len(w) for w in words)} chars")
print(words[:8])

### Build the vocabulary

**Why this step:** Before we can feed characters into a neural net, we must map every character to an integer index. We reserve index `0` for the special `.` token, which marks both the beginning and end of a name.

In [ ]:
stoi, itos, vocab_size = build_vocab(words)
print(itos)
print(f"Vocabulary size: {vocab_size}")

### Split the data and build tensors

**Why this step:** We split names into train / validation / test sets so we can detect overfitting. The `build_dataset` helper turns each name into a sequence of `(context, next_character)` examples. A sliding window of length `block_size` predicts the character that follows it.

**Why `block_size = 8`:** More context gives the model more signal, but it also increases model size if we flatten everything. The hierarchical architecture below lets us increase context without blowing up the first-layer weight matrix.

In [ ]:
# Set a reproducible seed for the whole notebook
generator = set_seed(42)

# Split the list of words
train_words, val_words, test_words = split_dataset(words, train_frac=0.8, val_frac=0.1, shuffle=True, seed=42)

block_size = 8  # context length: how many characters do we take to predict the next one?

Xtr,  Ytr  = build_dataset(train_words, stoi, block_size=block_size, generator=generator)
Xdev, Ydev = build_dataset(val_words,   stoi, block_size=block_size, generator=generator)
Xte,  Yte  = build_dataset(test_words,  stoi, block_size=block_size, generator=generator)

print('Train:    ', Xtr.shape, Ytr.shape)
print('Validate: ', Xdev.shape, Ydev.shape)
print('Test:     ', Xte.shape, Yte.shape)

In [ ]:
for x, y in zip(Xtr[:20], Ytr[:20]):
    print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

## The tiny layers we will use

In earlier lectures we wrote these layers by hand. They are intentionally small so you can read every line. Now that we understand how they work, we import them from `nnzero` and focus on how to *compose* them into a hierarchical model.

**Why Kaiming initialization:** In a deep network, the variance of activations can explode or shrink at each layer. Kaiming init sets weight scale to `1 / sqrt(fan_in)`, which keeps activations roughly unit variance layer after layer.

**Why BatchNorm:** Even with good initialization, internal covariate shift can destabilize training. BatchNorm normalizes each mini-batch to zero mean and unit variance, then learns a scale (`gamma`) and shift (`beta`). This lets us use higher learning rates and train deeper models.

**Why `FlattenConsecutive`:** This is the key layer that makes the network hierarchical. It groups consecutive time steps in the sequence and merges their embeddings into a single feature vector, so higher layers operate on larger and larger receptive fields.

In [ ]:
# Import the tiny layers from the course package.
# If you want to see the from-scratch implementations, open nnzero/layers.py.
from nnzero import Linear, BatchNorm1d, Tanh, Embedding, FlattenConsecutive, Sequential

print("Layers ready:", Linear, BatchNorm1d, Tanh, Embedding, FlattenConsecutive, Sequential)

In [ ]:
# Re-seed so the model weights are reproducible across runs.
torch.manual_seed(42)

### Build the hierarchical model

The architecture below is a toy WaveNet: embeddings are grouped in pairs, passed through a linear layer, batch-normalized, and activated with `tanh`. Each stage doubles the receptive field while keeping the parameter count modest.

In [ ]:
n_embd = 24   # dimensionality of the character embedding vectors
n_hidden = 128  # number of neurons in each hidden layer

model = Sequential([
    Embedding(vocab_size, n_embd),
    FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size),
])

# Scale down the final layer so the model starts with less confident logits.
# This improves initial cross-entropy loss because random logits are closer to uniform.
with torch.no_grad():
    model.layers[-1].weight *= 0.1

parameters = model.parameters()
print(f"Total parameters: {sum(p.nelement() for p in parameters):,}")
for p in parameters:
    p.requires_grad = True

### 🏋️ Try it yourself #1: understand the receptive field

The model above uses three `FlattenConsecutive(2)` layers. After the first layer, each output sees 2 characters; after the second, 4; after the third, 8.

Write a short snippet that prints the shape of `model.layers[i].out` for each layer when you forward a single batch of `block_size=8` characters. Confirm that the final logits have shape `(batch_size, vocab_size)`.

In [ ]:
# Your code here

### Solution

In [ ]:
# Forward one example through the model and inspect intermediate shapes.
model.eval()  # disable batch-norm updates for this inspection
with torch.no_grad():
    xb = Xtr[:4]  # small batch
    out = xb
    for i, layer in enumerate(model.layers):
        out = layer(out)
        print(f"Layer {i:2d} ({layer.__class__.__name__:20s}) output shape: {tuple(out.shape)}")
    print(f"Final logits shape: {tuple(out.shape)}")

## Training

**Why cross-entropy:** For classification, cross-entropy measures the difference between the model's predicted probability distribution and the true one-hot target. It is the natural loss when the final layer outputs logits and we want to maximize the log-likelihood of the next character.

**Why SGD with learning-rate decay:** We use plain stochastic gradient descent because the problem is small and the gradient noise helps escape sharp minima. Decaying the learning rate after most of the progress has been made lets the model settle into a flatter minimum.

In [ ]:
max_steps = 1000 if QUICK_RUN else 200_000
batch_size = 32
lossi = []

# Set eval=False on all layers with a training flag so BatchNorm updates its running stats.
# (At the start of training every layer is in training mode by default.)
for layer in model.layers:
    if hasattr(layer, 'training'):
        layer.training = True

for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=generator)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update: simple SGD with step learning-rate decay
    lr = 0.1 if i < 150_000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    if i % 10_000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))
plt.xlabel('Step (x1000)')
plt.ylabel('log10(loss)')
plt.title('Smoothed training loss')
plt.grid(True)
plt.show()

## Evaluation

**Why switch to evaluation mode:** BatchNorm keeps running estimates of mean and variance during training. At test time we must use those running statistics instead of the mini-batch statistics, otherwise the validation loss will be unreliable.

In [ ]:
# Put layers into eval mode (needed for BatchNorm especially)
for layer in model.layers:
    if hasattr(layer, 'training'):
        layer.training = False

In [ ]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    print(f"{split:6s}: {loss.item():.4f}")

split_loss('train')
split_loss('val')
split_loss('test')

### Performance log

These numbers are for reference from the original lecture run:

- original (3 character context + 200 hidden neurons, 12K params): train 2.058, val 2.105
- context: 3 -> 8 (22K params): train 1.918, val 2.027
- flat -> hierarchical (22K params): train 1.941, val 2.029
- fix bug in batchnorm: train 1.912, val 2.022
- scale up the network: n_embd 24, n_hidden 128 (76K params): train 1.769, val 1.993

Your numbers may differ slightly because of the modernized data pipeline and explicit seeds.

## Sampling new names

We feed the model a context of all `.` tokens and repeatedly sample the next character until the model emits another `.`.

In [ ]:
# sample from the model
for _ in range(20):
    out = []
    context = [0] * block_size  # initialize with all '.' tokens
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=generator).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

### 🏋️ Try it yourself #2: tune the model

Experiment with one hyperparameter change and re-run the training cell above. For example:

- Change `block_size` to 4 or 16 and adjust the number of `FlattenConsecutive(2)` stages accordingly.
- Change `n_hidden` or `n_embd`.
- Add a fourth hierarchical stage.

Record the validation loss you obtain. Which change helps most?

In [ ]:
# Your code here (copy the model definition and training cells, then edit)

### Solution

In [ ]:
# Example: increase context to 16 and add one more stage.
# block_size = 16
# model = Sequential([
#     Embedding(vocab_size, n_embd),
#     FlattenConsecutive(2), Linear(n_embd*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#     FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#     FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#     FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#     Linear(n_hidden, vocab_size),
# ])
# Then re-run the training and evaluation cells.

### Preview: Why convolutions?

The hierarchical model is already doing something convolution-like: the same `Linear` transformation is applied to many local patches of the input. The preview below shows how a for-loop over contexts can be replaced by a single batched forward pass, which is the key insight behind 1-D convolutions.

In [ ]:
for x, y in zip(Xtr[7:15], Ytr[7:15]):
    print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

In [ ]:
# Forward a single example
logits = model(Xtr[[7]])
print(logits.shape)

In [ ]:
# Forward all of them one at a time
logits = torch.zeros(8, 27)
for i in range(8):
    logits[i] = model(Xtr[[7+i]])
print(logits.shape)

In [ ]:
# A convolution is a "for loop" that lets us forward Linear layers efficiently over space.
# In the next lecture we will replace these loops with torch.nn.Conv1d.